# Statistical Cox

In [ ]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def make_sksurv_target(duration, event):
    return np.array([(bool(e), d) for e, d in zip(event, duration)],
                    dtype=[('event', 'bool'), ('duration', 'float')])

In [ ]:
split_ratio = 0.3
slice_num = 5
batch_size = 32
lr = 0.0005
epochs = 50
tabular_only = False

In [ ]:
tab_df = pd.read_csv("dataset/tabular_input.csv", parse_dates=["LM_DIAG_date","os_date","collection_date"])

In [ ]:
trainset = tab_df[tab_df['split_label'] == 1]
testset = tab_df[tab_df['split_label'] == 0]

In [ ]:
features = [c for c in tab_df.columns if c not in ['id', 'LM_DIAG_date', 'os_date','collection_date','split_label', 'prog_risk','time_len','c_date_pos','mri_84','mic_84']]
train_x = trainset[features]
test_x = testset[features]

### Model Running

In [ ]:
cph = CoxPHFitter(penalizer=0.0001)
cph.fit(train_x, duration_col='duration', event_col='os_status')

In [ ]:
train_c_index = cph.concordance_index_
print(f"训练集 C-index: {train_c_index:.4f}")
test_c_index = cph.score(testset, scoring_method="concordance_index")
print(f"测试集 C-index: {test_c_index:.4f}")

In [ ]:
cph.plot()

In [ ]:
cph.summary